In [0]:
import os
import mlflow

os.environ["MLFLOW_DFS_TMP"] = "/Volumes/teams/sensorx/data-dump/mlflow_tmp"

model = mlflow.spark.load_model("models:/teams.sensorx.mlp_fault_prediction/3")

In [0]:
from pyspark.ml.feature import StandardScaler
from pyspark.ml.functions import vector_to_array
import pyspark.sql.functions as F

os.environ["MLFLOW_DFS_TMP"] = "/Volumes/teams/sensorx/data-dump/mlflow_tmp"

# Read test data and normalize (model was trained on normalized features)
train = spark.read.table("teams.sensorx.train_data")
test = spark.read.table("teams.sensorx.test_data")

scaler = StandardScaler(inputCol="features", outputCol="features_scaled", withMean=True, withStd=True)
scaler_model = scaler.fit(train)
test_norm = scaler_model.transform(test).drop("features").withColumnRenamed("features_scaled", "features")

# Filter to latest 3 weeks
max_date = test_norm.agg(F.max("timeStamp")).collect()[0][0]
test_recent = test_norm.filter(F.col("timeStamp") >= F.date_sub(F.lit(max_date), 21))

# Load model version 3 (matches 161-feature test data)
model_v3 = mlflow.spark.load_model("models:/teams.sensorx.mlp_fault_prediction/3")
predictions = model_v3.transform(test_recent)

# Join with original table to get actual fault state and deviceId
original = spark.read.table("teams.sensorx.df_sx_800_with_delta").select("timeStamp", "payload_fault_state", "properties_deviceId")

pred_with_fault = predictions.select(
    "timeStamp", "label", "prediction"
).join(original, on="timeStamp", how="left") \
 .withColumn("date", F.to_date("timeStamp"))

# One prediction per day per device
daily_results = pred_with_fault.groupBy("date", "properties_deviceId").agg(
    F.max(F.col("prediction").cast("int")).alias("daily_prediction"),
    F.max(F.col("payload_fault_state").cast("int")).alias("actual_fault_state"),
    F.max("label").alias("horizon_failure"),
    F.count("*").alias("readings_count")
).orderBy("date", "properties_deviceId")

display(daily_results)

In [0]:
from pyspark.sql import Row

data = [
    Row(Trial=1,  AUC=0.6525), #Logistic Regression 1, 5.mars
    Row(Trial=2,  AUC=0.6251), #Random Forest 1, 5.mars
    Row(Trial=3,  AUC=0.5383), #Gradient Boosted Trees 1, 5.mars
    Row(Trial=4,  AUC=0.9115), #Logistic Regression 2, 8.mars
    Row(Trial=5,  AUC=0.9139), #Logistic Regression 3, 8.mars
    Row(Trial=6,  AUC=0.9147), #Gradient Boosted Trees 2, 8.mars
    Row(Trial=7,  AUC=0.9114), #Logistic Regression 4, 12.mars
    Row(Trial=8,  AUC=0.9005), #Logistic Regression 5, 12.mars
    Row(Trial=9,  AUC=0.9237), #Logistic Regression 6, 12.mars
    Row(Trial=10, AUC=0.9350), #Logistic Regression 7, 12.mars
    Row(Trial=11, AUC=0.9411), #Random Forest 2, 12.mars
    Row(Trial=12, AUC=0.9156), #Gradient Boosted Trees 3, 12.mars
    Row(Trial=13, AUC=0.9240), #Random Forest 3, 15.mars
    Row(Trial=14, AUC=0.9360), #Gradient Boosted Trees 5, 17.mars
    Row(Trial=15, AUC=0.9379), #Multilayer Perceptron 1, 23.mars
    Row(Trial=16, AUC=0.9238), #Multilayer Perceptron 2, 29.mars
    Row(Trial=17, AUC=0.9136), #Multilayer Perceptron 3, 29.mars
    Row(Trial=18, AUC=0.9137), #Multilayer Perceptron 4, 29.mars
    Row(Trial=19, AUC=0.9034), #Multilayer Perceptron 5, 29.mars
    Row(Trial=20, AUC=0.8837), #Multilayer Perceptron 6, 29.mars
    Row(Trial=21, AUC=0.8847), #Multilayer Perceptron 7, 29.mars
    Row(Trial=22, AUC=0.9314), #Multilayer Perceptron 8, 29.mars
    Row(Trial=23, AUC=0.9452), #Multilayer Perceptron 9, 2.apríl
]

df_auc = spark.createDataFrame(data)
display(df_auc)

Databricks visualization. Run in Databricks to view.

In [0]:
import plotly.graph_objects as go

rows = df_auc.orderBy("Trial").collect()
trials = [r.Trial for r in rows]
aucs = [r.AUC for r in rows]

fig = go.Figure()
fig.add_trace(go.Scatter(x=trials, y=aucs, mode="lines+markers", name="AUC", line=dict(color="#919191")))

# Lines with information about where LR and MLP stop/start
fig.add_vline(x=10, line_dash="dash", line_color="#BF7080",
              annotation_text="LR testing stops", annotation_position="top left")
fig.add_vline(x=16, line_dash="dash", line_color="#99DDB4",
              annotation_text="MLP testing starts", annotation_position="top left")

fig.update_layout(
    title="AUC vs Trials",
    xaxis_title="Trial",
    yaxis_title="AUC",
    plot_bgcolor="white",
    paper_bgcolor="white",
    xaxis=dict(showgrid=True, gridcolor="lightgray"),
    yaxis=dict(showgrid=True, gridcolor="lightgray"),
)
fig.show()


## MLP Fault Prediction Model — Results & Evaluation
This section presents the evaluation of the **Multilayer Perceptron (MLP)** classifier trained to predict X-ray controller faults using sensor telemetry data. The model uses a **failure horizon** label (fault within the next N hours) rather than the instantaneous fault state, enabling **early warning** capability.

In [0]:
from pyspark.sql import functions as F, Window
import pandas
from datetime import datetime, timedelta
from pyspark.sql.window import Window
import os
import mlflow
from marel.services import ServiceProvider
from marel.databricks import DatabricksProvider
provider = DatabricksProvider()

In [0]:
df_csv = spark.read.format("csv") \
    .option("header", True) \
    .option("inferSchema", True) \
    .option("multiLine", True) \
    .option("escape", "\"") \
    .option("quote", "\"") \
    .option("delimiter", ",") \
    .load("/Volumes/teams/sensorx/data-dump/deviceID_Serial_GenSerial_machineCode.csv")

# Device IDs that have at least one 800W serial number
devices_800w = (
    df_csv.filter(F.col("gentype") == "800W")
    .select("properties_deviceId")
    .distinct()
)

devices_500w = (
    df_csv.filter(F.col("gentype") == "500W")
    .select("properties_deviceId")
    .distinct()
)

devices_800w.write.format("delta").mode("overwrite").saveAsTable("teams.sensorx.devices_800w")

devices_500w.write.format("delta").mode("overwrite").saveAsTable("teams.sensorx.devices_500w")

In [0]:
from pyspark.ml.feature import VectorAssembler, StringIndexer, OneHotEncoder

def process_data_800W(months):
    df_sx_telemetry = spark.table('marel_digital_machines.sensorx.marel_sensorx_telemetry')
    df_sx_telemetry = df_sx_telemetry.filter(F.col('timeStamp') > F.add_months(F.current_date(),-months)).select(
        "timeStamp",
        "properties_deviceId",
        "payload_xrayController_filamentCurrent",
        "payload_xrayController_temperature",
        "payload_xrayController_tubeCurrent",
        "payload_xrayController_tubeVoltage",
        "payload_xrayController_onTime"
    )

    devices_800w= spark.table("teams.sensorx.devices_800w")

    # Filter telemetry to only 800W devices (broadcast — devices_800w is small)
    df_sx_telemetry = df_sx_telemetry.join(
        F.broadcast(devices_800w), "properties_deviceId", "inner"
    )

    df_sx_xraycontroller_fault = (
        spark.table('marel_digital_machines.sensorx.marel_sensorx_xraycontroller_property_fault')
        .filter(
            (F.col('timeStamp') > F.add_months(F.current_date(), -months)) &
            (F.col('payload_fault_faultName') == "faultRegulation")
        )
        .select("timeStamp", "properties_deviceId", "payload_fault_faultName", "payload_fault_state")
    )

    # Filter faults to 800W devices only
    df_sx_xraycontroller_fault = df_sx_xraycontroller_fault.join(
        F.broadcast(devices_800w), "properties_deviceId", "inner"
    )

    # --- Union + forward-fill ---
    telem_only = [c for c in df_sx_telemetry.columns
                if c not in ("timeStamp", "properties_deviceId")]

    telem_part = df_sx_telemetry.select(
        "timeStamp", "properties_deviceId",
        *telem_only,
        F.lit(None).cast("string").alias("payload_fault_faultName"),
        F.lit(None).cast("boolean").alias("payload_fault_state"),
        F.lit(True).alias("_is_telem"),
    )

    fault_part = df_sx_xraycontroller_fault.select(
        "timeStamp", "properties_deviceId",
        *[F.lit(None).cast(dict(df_sx_telemetry.dtypes)[c]).alias(c)
        for c in telem_only],
        "payload_fault_faultName",
        "payload_fault_state",
        F.lit(False).alias("_is_telem"),
    )

    w = Window.partitionBy("properties_deviceId").orderBy("timeStamp")

    df = (
        telem_part.unionByName(fault_part)
        .withColumn("payload_fault_faultName",
                    F.last("payload_fault_faultName", ignorenulls=True).over(w))
        .withColumn("payload_fault_state",
                    F.last("payload_fault_state", ignorenulls=True).over(w))
        .filter(F.col("_is_telem"))
        .drop("_is_telem")
    )

    df = df.withColumn(
        "payload_fault_state", F.coalesce(F.col("payload_fault_state"), F.lit(False))
    ).withColumn(
        "payload_fault_faultName", F.coalesce(F.col("payload_fault_faultName"), F.lit("faultRegulation"))
    )

    # Compute delta_seconds
    order_cols = [F.col("timeStamp").asc()]
    if "serialNumber" in df.columns:
        order_cols.append(F.col("serialNumber").asc())
    w = Window.partitionBy("properties_deviceId").orderBy(*order_cols)

    df = (
        df
        .withColumn("prev_ts", F.lag("timeStamp").over(w))
        .withColumn(
            "delta_seconds",
            F.when(F.col("prev_ts").isNull(), F.lit(None).cast("double"))
            .otherwise(
                F.when(
                    (F.col("timeStamp").cast("long") - F.col("prev_ts").cast("long")) < 0,
                    None
                ).otherwise(F.col("timeStamp").cast("long") - F.col("prev_ts").cast("long"))
            )
        )
        .drop("prev_ts")
    )

    df = df.drop("serialNumber", "payload_fault_faultName", "GeneratorType")

    # ---- Feature engineering (matching (Clone) model_testing) ----

    # Sensor columns = everything except identifiers and target
    sensor_cols = [c for c in df.columns
                   if c not in {"properties_deviceId", "timeStamp", "payload_fault_state"}]

    # Drop rows with null sensor values
    df = df.na.drop(subset=sensor_cols)

    # One-Hot Encode properties_deviceId
    indexer = StringIndexer(inputCol="properties_deviceId", outputCol="deviceId_index")
    df_indexed = indexer.fit(df).transform(df)

    encoder = OneHotEncoder(inputCol="deviceId_index", outputCol="deviceId_ohe")
    df_encoded = encoder.fit(df_indexed).transform(df_indexed)

    # Lag features (n_lags = 20)
    n_lags = 20
    w_lag = Window.partitionBy("properties_deviceId").orderBy("timeStamp")

    for L in range(1, n_lags + 1):
        for col_name in sensor_cols:
            df_encoded = df_encoded.withColumn(f"{col_name}{L}", F.lag(col_name, L).over(w_lag))

    lag_cols = [f"{col}{L}" for L in range(1, n_lags + 1) for col in sensor_cols]

    # Drop rows with null lag values
    df_clean = df_encoded.na.drop(subset=lag_cols)

    # Assemble features: sensor + lag + OHE device ID
    feature_input_cols = sensor_cols + lag_cols + ["deviceId_ohe"]
    assembler = VectorAssembler(inputCols=feature_input_cols, outputCol="features")
    df_features = assembler.transform(df_clean)

    df_features = df_features.select("timeStamp", "properties_deviceId", "features")

    return df_features

In [0]:
months = 3

df = process_data_800W(months)
display(df.limit(10))

In [0]:
from pyspark.ml.feature import StandardScaler
from pyspark.ml.functions import vector_to_array

os.environ["MLFLOW_DFS_TMP"] = "/Volumes/teams/sensorx/data-dump/mlflow_tmp"

# Read existing test/train data
train = spark.read.table("teams.sensorx.train_features_delta")
test = spark.read.table("teams.sensorx.test_features_delta")

# Normalize features (fit scaler on training data to avoid data leakage)
scaler = StandardScaler(inputCol="features", outputCol="features_scaled", withMean=True, withStd=True)
scaler_model = scaler.fit(train)
test_norm = scaler_model.transform(test).drop("features").withColumnRenamed("features_scaled", "features")

# Load model and run predictions
model = mlflow.spark.load_model("models:/teams.sensorx.mlp_fault_prediction/3")
predictions = model.transform(test_norm)

# Simplify output: extract fault probability and drop vector columns
results = predictions.select(
    "timeStamp",
    F.col("label").alias("actual_label"),
    F.col("prediction").cast("int").alias("predicted_label"),
    F.round(vector_to_array("probability")[1], 4).alias("fault_probability"),
)

display(results)

In [0]:
%pip install shap -q

In [0]:
# === Maintenance Decision Table with SHAP Explanations ===
import shap
import numpy as np
from pyspark.ml.feature import StandardScaler
from pyspark.ml.functions import vector_to_array

# -- Reconstruct feature names from the feature-engineering pipeline (cell 8) --
sensor_cols = [
    "payload_xrayController_filamentCurrent",
    "payload_xrayController_temperature",
    "payload_xrayController_tubeCurrent",
    "payload_xrayController_tubeVoltage",
    "payload_xrayController_onTime",
    "delta_seconds",
]
n_lags = 20
lag_cols = [f"{col}{L}" for L in range(1, n_lags + 1) for col in sensor_cols]
n_ohe = 161 - len(sensor_cols) - len(lag_cols)
ohe_cols = [f"deviceId_ohe_{i}" for i in range(n_ohe)]
feature_names = sensor_cols + lag_cols + ohe_cols
assert len(feature_names) == 161, f"Expected 161 features, got {len(feature_names)}"

def friendly_name(name):
    """Convert technical feature names to human-readable labels."""
    name = name.replace("payload_xrayController_", "")
    if name.startswith("deviceId_ohe_"):
        return None
    for base in ["filamentCurrent", "temperature", "tubeCurrent",
                 "tubeVoltage", "onTime", "delta_seconds"]:
        if name.startswith(base) and name != base:
            lag = name[len(base):]
            if lag.isdigit():
                return f"{base} (lag {lag})"
    return name

# -- Normalize and predict using df from process_data (cell 9) --
# (teams.sensorx.test/train_features_delta tables no longer exist)
scaler = StandardScaler(inputCol="features", outputCol="features_scaled", withMean=True, withStd=True)
scaler_model = scaler.fit(df)
df_norm = scaler_model.transform(df).drop("features").withColumnRenamed("features_scaled", "features")
predictions = model.transform(df_norm)

# -- One representative feature vector per device (highest prob, last 24 h) --
pred_full = predictions.select(
    "timeStamp", "properties_deviceId", "features",
    vector_to_array("probability")[1].alias("fault_probability"),
)

max_ts = pred_full.agg(F.max("timeStamp")).collect()[0][0]
recent = pred_full.filter(
    F.col("timeStamp") >= F.lit(max_ts) - F.expr("INTERVAL 24 HOURS")
)

w_rank = Window.partitionBy("properties_deviceId").orderBy(F.col("fault_probability").desc())
device_reps = (
    recent.withColumn("rn", F.row_number().over(w_rank))
    .filter(F.col("rn") == 1).drop("rn")
    .select("properties_deviceId", "fault_probability",
            vector_to_array("features").alias("feat_arr"))
    .toPandas()
)

X = np.array(device_reps["feat_arr"].tolist())

# -- Background dataset for SHAP --
bg = np.array(
    df_norm.select(vector_to_array("features").alias("f"))
    .limit(100).toPandas()["f"].tolist()
)

# -- Predict function (fast numpy path, Spark fallback) --
try:
    _layers = model.getLayers()
    _raw_w = np.array(model.weights.toArray())
    _params, _off = [], 0
    for i in range(len(_layers) - 1):
        n_in, n_out = _layers[i], _layers[i + 1]
        W = _raw_w[_off : _off + n_in * n_out].reshape(n_in, n_out)
        _off += n_in * n_out
        b = _raw_w[_off : _off + n_out]
        _off += n_out
        _params.append((W, b))

    def predict_fn(X_in):
        h = np.atleast_2d(X_in).astype(float)
        for i, (W, b) in enumerate(_params):
            h = h @ W + b
            if i < len(_params) - 1:
                h = 1.0 / (1.0 + np.exp(-np.clip(h, -500, 500)))
        e = np.exp(h - h.max(axis=1, keepdims=True))
        return (e / e.sum(axis=1, keepdims=True))[:, 1]
    print(f"Using fast local numpy predict (layers: {_layers})")
except Exception as exc:
    from pyspark.ml.linalg import Vectors as _Vec
    from pyspark.sql import Row as _Row
    print(f"Weights unavailable ({exc}); using Spark predict (slower)")
    def predict_fn(X_in):
        X_in = np.atleast_2d(X_in)
        rows = [_Row(features=_Vec.dense(r.tolist())) for r in X_in]
        preds = model.transform(spark.createDataFrame(rows))
        return preds.select(
            vector_to_array("probability")[1].alias("p")
        ).toPandas()["p"].values

# -- Compute SHAP values --
print(f"Computing SHAP values for {len(X)} devices ...")
explainer = shap.KernelExplainer(predict_fn, shap.sample(bg, 50))
shap_values = explainer.shap_values(X, nsamples="auto", silent=True)
print("SHAP computation complete")

# -- Top-3 reason string per device --
def top3_reason(sv):
    scored = []
    for j, val in enumerate(sv):
        fn = friendly_name(feature_names[j])
        if fn is not None:
            scored.append((fn, val))
    scored.sort(key=lambda x: abs(x[1]), reverse=True)
    parts = []
    for name, v in scored[:3]:
        arrow = "UP" if v > 0 else "DOWN"
        parts.append(f"{name} ({arrow} {abs(v):.4f})")
    return "; ".join(parts)

device_reps["main_reason"] = [
    top3_reason(shap_values[i]) for i in range(len(device_reps))
]

# -- Assemble final maintenance table --
result = device_reps[["properties_deviceId", "fault_probability", "main_reason"]].copy()
result.columns = ["machine_id", "failure_probability", "main_reason"]
result["risk_level"] = result["failure_probability"].apply(
    lambda p: "High" if p >= 0.80 else ("Medium" if p >= 0.50 else "Low")
)
result["recommended_action"] = result["failure_probability"].apply(
    lambda p: "Inspect immediately" if p >= 0.80
    else ("Schedule maintenance" if p >= 0.50 else "Continue monitoring")
)
result = (
    result[["machine_id", "failure_probability", "risk_level",
            "recommended_action", "main_reason"]]
    .sort_values("failure_probability", ascending=False)
    .reset_index(drop=True)
)

maintenance_table = spark.createDataFrame(result)
display(maintenance_table)

In [0]:
# === Top 10 Highest-Priority Machines ===
top10 = result.head(10)

print("=" * 100)
print("  TOP 10 HIGHEST-PRIORITY MACHINES — MAINTENANCE ACTION REQUIRED")
print("=" * 100)

for i, (_, row) in enumerate(top10.iterrows(), 1):
    print(f"\n  {i}. Machine:  {row['machine_id']}")
    print(f"     Risk:     {row['risk_level']} ({row['failure_probability']:.2%})")
    print(f"     Action:   {row['recommended_action']}")
    print(f"     Drivers:  {row['main_reason']}")

print("\n" + "=" * 100)
print("  Model: MLP Fault Prediction v3  |  Explanations: SHAP (top 3 features)")
print("=" * 100)

In [0]:
# === Visual Summary of Maintenance Decisions ===
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# --- Chart 1: Failure Probability per Machine (color-coded by risk) ---
df_plot = result.copy()
df_plot["machine_short"] = df_plot["machine_id"].str[:8] + "..."
df_plot = df_plot.sort_values("failure_probability", ascending=True)  # bottom-to-top

color_map = {"High": "#D94040", "Medium": "#E8A838", "Low": "#4CAF82"}
colors = [color_map[r] for r in df_plot["risk_level"]]

fig1 = go.Figure()
fig1.add_trace(go.Bar(
    x=df_plot["failure_probability"],
    y=df_plot["machine_short"],
    orientation="h",
    marker_color=colors,
    text=[f"{p:.1%}" for p in df_plot["failure_probability"]],
    textposition="outside",
    hovertext=[
        f"Machine: {mid}<br>Risk: {rl}<br>Probability: {fp:.4%}<br>Action: {ra}"
        for mid, rl, fp, ra in zip(
            df_plot["machine_id"], df_plot["risk_level"],
            df_plot["failure_probability"], df_plot["recommended_action"]
        )
    ],
    hoverinfo="text",
))

# Add risk level threshold lines
fig1.add_vline(x=0.80, line_dash="dash", line_color="#D94040", line_width=1,
               annotation_text="High (80%)", annotation_position="top")
fig1.add_vline(x=0.50, line_dash="dash", line_color="#E8A838", line_width=1,
               annotation_text="Medium (50%)", annotation_position="top")

fig1.update_layout(
    title="Failure Probability by Machine (15-Day Horizon)",
    xaxis_title="Failure Probability",
    yaxis_title="Machine ID",
    xaxis=dict(range=[0, 1.12], tickformat=".0%", showgrid=True, gridcolor="#EEEEEE"),
    yaxis=dict(showgrid=False),
    plot_bgcolor="white",
    paper_bgcolor="white",
    height=500,
    margin=dict(l=100),
)
fig1.show()

